# Imbalanced Bootstrap Results

This notebook reads the outputs from `imbalanced_bootstrap_stringency.py` and plots how NN/NP downsampling affects RF LOOCV AUC.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns
except ImportError:
    sns = None

if sns is not None:
    sns.set_theme(style="whitegrid")
else:
    plt.style.use("ggplot")

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pyproject.toml").is_file() and (p / "analysis").is_dir())
OUTPUT_DIR = REPO_ROOT / "analysis" / "outputs"
RESULTS_PATH = OUTPUT_DIR / "imbalanced_bootstrap_results.csv"
CI_PATH = OUTPUT_DIR / "imbalanced_bootstrap_auc_ci.csv"
TRIALS_PATH = OUTPUT_DIR / "stringency_optuna_trials.csv"
if not TRIALS_PATH.exists():
    TRIALS_PATH = OUTPUT_DIR / "stringency_sweep_results.csv"

RESULTS_PATH, CI_PATH, TRIALS_PATH

Run the script first if these files do not exist:

```bash
python analysis/cohort_imbalance/imbalanced_bootstrap_stringency.py
```

In [ ]:
results = pd.read_csv(RESULTS_PATH)
ci = pd.read_csv(CI_PATH)
trials = pd.read_csv(TRIALS_PATH)

if "mean_selected_motifs" not in trials.columns and "n_selected_motifs" in trials.columns:
    trials["mean_selected_motifs"] = trials["n_selected_motifs"]
if "optimal_mean_selected_motifs" not in results.columns and "optimal_n_selected_motifs" in results.columns:
    results["optimal_mean_selected_motifs"] = results["optimal_n_selected_motifs"]

display(ci)
display(results)

In [ ]:
wanted_columns = [
    'sampled_label',
    'bootstrap_fraction',
    "optimal_mean_selected_motifs",
    "optimal_full_bootstrap_selected_motifs",
]

results[wanted_columns]

In [ ]:
results.groupby(["sampled_label", "bootstrap_fraction"])

In [ ]:
optimal_vs_full = results.groupby(["sampled_label", "bootstrap_fraction"])[["optimal_fold_union_discriminative_motifs_NN", "optimal_fold_union_discriminative_motifs_NP"]].mean()
optimal_vs_full


In [ ]:
mean_selected_vs_sample_fraction = results.groupby("bootstrap_fraction")["optimal_mean_selected_motifs"].mean()
mean_selected_vs_sample_fraction.plot()

## AUC Confidence Intervals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for sampled_label, group in ci.groupby("sampled_label"):
    group = group.sort_values("bootstrap_fraction")
    ax.plot(group["bootstrap_fraction"], group["mean_auc"], marker="o", label=f"sampled {sampled_label}")
    ax.fill_between(
        group["bootstrap_fraction"],
        group["ci_low"],
        group["ci_high"],
        alpha=0.2,
    )
ax.set_xlabel("Fraction of sampled-label patients kept")
ax.set_ylabel("RF LOOCV AUC")
ax.set_ylim(0, 1.02)
ax.set_title("AUC confidence interval across bootstraps")
ax.legend(title="Experiment")
plt.show()

## Per-Bootstrap AUC Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
if sns is not None:
    sns.stripplot(
        data=results,
        x="bootstrap_fraction",
        y="optimal_auc",
        hue="sampled_label",
        dodge=True,
        alpha=0.75,
        ax=ax,
    )
else:
    for sampled_label, group in results.groupby("sampled_label"):
        ax.scatter(group["bootstrap_fraction"], group["optimal_auc"], label=sampled_label, alpha=0.75)
ax.set_xlabel("Fraction of sampled-label patients kept")
ax.set_ylabel("Best RF LOOCV AUC")
ax.set_ylim(0, 1.02)
ax.set_title("Bootstrap-level AUC values")
ax.legend(title="Sampled label")
plt.show()

## Optimal Stringency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for sampled_label, group in ci.groupby("sampled_label"):
    group = group.sort_values("bootstrap_fraction")
    ax.plot(
        group["bootstrap_fraction"],
        group["mean_optimal_stringency"],
        marker="o",
        label=f"sampled {sampled_label}",
    )
ax.set_xlabel("Fraction of sampled-label patients kept")
ax.set_ylabel("Mean optimal stringency")
ax.set_ylim(0, 1.02)
ax.set_title("Selected patient-prevalence stringency")
ax.legend(title="Experiment")
plt.show()

## Optuna Stringency Trial Detail

In [ ]:
trial_summary = (
    trials.groupby(["sampled_label", "bootstrap_fraction", "stringency"], observed=True)
    .agg(mean_auc=("roc_auc", "mean"), mean_selected_motifs=("mean_selected_motifs", "mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
for (sampled_label, fraction), group in trial_summary.groupby(["sampled_label", "bootstrap_fraction"]):
    group = group.sort_values("stringency")
    ax.plot(group["stringency"], group["mean_auc"], marker="o", label=f"{sampled_label} {fraction:.1f}")
ax.set_xlabel("Stringency")
ax.set_ylabel("Mean RF LOOCV AUC")
ax.set_ylim(0, 1.02)
ax.set_title("AUC across Optuna stringency trials")
ax.legend(title="Sampled group", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

display(trial_summary.head(20))

## Discriminative Motifs vs Stringency

This section counts hard-discriminative motifs across a fixed stringency grid for each bootstrap and each motif-associated class. It is independent of the Optuna-selected best stringency and does not rerun RF, LOOCV classification, or Optuna.


In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = REPO_ROOT
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.cohort_imbalance.imbalanced_bootstrap_stringency import (
    DEFAULT_ARTIFACT_PATH,
    discover_hard_discriminative_motifs,
    load_artifact,
    load_patient_classes,
)
from cism.cism import DiscriminativeMotifs

config_path = OUTPUT_DIR / 'imbalanced_bootstrap_config.json'
config = json.loads(config_path.read_text()) if config_path.exists() else {}
labels = config.get('labels', ['NN', 'NP'])
p_value_threshold = float(config.get('fanmod_p_value_threshold', 0.05))
artifact_path = Path(config.get('artifact_path', DEFAULT_ARTIFACT_PATH))
artifact = load_artifact(artifact_path)
discriminator = artifact['discriminator']
patient_classes = load_patient_classes(discriminator, labels)
motifs_df = discriminator.cism.motifs_dataset
stringency_grid = [round(x / 10, 1) for x in range(0, 11)]

count_rows = []
for _, result_row in results.iterrows():
    sampled_patients = [p for p in str(result_row['sampled_patients']).split('|') if p and p != 'nan']
    other_label = result_row['other_label']
    other_patients = patient_classes.index[
        patient_classes[DiscriminativeMotifs.PATIENT_CLASS] == other_label
    ].tolist()
    cohort_patients = sampled_patients + other_patients
    bootstrap_summary = discover_hard_discriminative_motifs(
        motifs_df=motifs_df,
        patient_classes=patient_classes,
        cohort_patients=cohort_patients,
        p_value_threshold=p_value_threshold,
    )
    for stringency in stringency_grid:
        passing = bootstrap_summary[bootstrap_summary['patient_percentage'] >= stringency]
        for motif_label in labels:
            count_rows.append({
                'sampled_label': result_row['sampled_label'],
                'bootstrap_fraction': result_row['bootstrap_fraction'],
                'bootstrap_index': result_row['bootstrap_index'],
                'sampled_patients': result_row['sampled_patients'],
                'stringency': stringency,
                'motif_label': motif_label,
                'discriminative_motifs_passing': int((passing[DiscriminativeMotifs.PATIENT_CLASS] == motif_label).sum()),
            })

motifs_vs_stringency = pd.DataFrame(count_rows)
motifs_vs_stringency_path = OUTPUT_DIR / 'discriminative_motifs_vs_stringency_by_class.csv'
motifs_vs_stringency.to_csv(motifs_vs_stringency_path, index=False)
print(f'Saved motif-count table: {motifs_vs_stringency_path}')
display(motifs_vs_stringency.head())


In [ ]:
motif_count_summary = (
    motifs_vs_stringency.groupby(
        ['sampled_label', 'bootstrap_fraction', 'stringency', 'motif_label'],
        as_index=False,
        observed=True,
    )
    .agg(mean_discriminative_motifs=('discriminative_motifs_passing', 'mean'))
)

grid = motif_count_summary.groupby(['sampled_label', 'motif_label'], observed=True)
for (sampled_label, motif_label), group in grid:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_group = group.copy()
    plot_group['mean_discriminative_motifs'] = plot_group['mean_discriminative_motifs'].clip(lower=0.1)

    if sns is not None:
        sns.lineplot(
            data=plot_group,
            x='stringency',
            y='mean_discriminative_motifs',
            hue='bootstrap_fraction',
            marker='o',
            palette='viridis',
            ax=ax,
        )
    else:
        for fraction, fraction_group in plot_group.groupby('bootstrap_fraction'):
            ax.plot(
                fraction_group['stringency'],
                fraction_group['mean_discriminative_motifs'],
                marker='o',
                label=f'{fraction:.1f}',
            )

    ax.set_yscale('log')
    ax.set_title(f'{motif_label}-associated motifs when sampling {sampled_label}')
    ax.set_xlabel('Stringency')
    ax.set_ylabel('Average hard-discriminative motifs passing threshold')
    # ...existing code...
    ax.set_yscale('log')
    ax.set_ylim(bottom=1)
    ax.set_title(f'{motif_label}-associated motifs when sampling {sampled_label}')
    ax.set_xlabel('Stringency')
    ax.set_ylabel('Average hard-discriminative motifs passing threshold')
# ...existing code...
    ax.legend(title='Bootstrap fraction', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.show()

display(motif_count_summary.head(20))

In [ ]:
motif_count_summary = (
    motifs_vs_stringency.groupby(
        ['sampled_label', 'bootstrap_fraction', 'stringency', 'motif_label'],
        as_index=False,
        observed=True,
    )
    .agg(mean_discriminative_motifs=('discriminative_motifs_passing', 'mean'))
)

grid = motif_count_summary.groupby(['sampled_label', 'motif_label'], observed=True)
for (sampled_label, motif_label), group in grid:
    fig, ax = plt.subplots(figsize=(8, 5))
    if sns is not None:
        sns.lineplot(
            data=group,
            x='stringency',
            y='mean_discriminative_motifs',
            hue='bootstrap_fraction',
            marker='o',
            palette='viridis',
            ax=ax,
        )
    else:
        for fraction, fraction_group in group.groupby('bootstrap_fraction'):
            ax.plot(fraction_group['stringency'], fraction_group['mean_discriminative_motifs'], marker='o', label=f'{fraction:.1f}')
    ax.set_title(f'{motif_label}-associated motifs when sampling {sampled_label}')
    ax.set_xlabel('Stringency')
    ax.set_ylabel('Average hard-discriminative motifs passing threshold')
    ax.legend(title='Bootstrap fraction', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.show()

display(motif_count_summary.head(20))


## Original Feature Overlap

This section compares each bootstrap's hard-discriminative features at its optimal stringency with the original full-cohort hard-discriminative feature set. It uses saved bootstrap results and the serialized artifact only; it does not rerun RF, LOOCV, or Optuna.

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = REPO_ROOT
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.cohort_imbalance.imbalanced_bootstrap_stringency import (
    DEFAULT_ARTIFACT_PATH,
    discover_hard_discriminative_motifs,
    load_artifact,
    load_patient_classes,
    select_motifs_by_label,
)
from cism.cism import DiscriminativeMotifs

config_path = OUTPUT_DIR / 'imbalanced_bootstrap_config.json'
config = json.loads(config_path.read_text()) if config_path.exists() else {}
labels = config.get('labels', ['NN', 'NP'])
max_features = int(config.get('max_features', 30))
p_value_threshold = float(config.get('fanmod_p_value_threshold', 0.05))
artifact_path = Path(config.get('artifact_path', DEFAULT_ARTIFACT_PATH))

artifact = load_artifact(artifact_path)
discriminator = artifact['discriminator']
patient_classes = load_patient_classes(discriminator, labels)
motifs_df = discriminator.cism.motifs_dataset
best_feature_conf = artifact.get('best_feature_conf')
original_stringency = float(getattr(best_feature_conf, 'shared_fractionage', config.get('original_stringency', 0.4404145064107466)))
original_max_features = int(getattr(best_feature_conf, 'max_class_features', max_features))

original_summary = discover_hard_discriminative_motifs(
    motifs_df=motifs_df,
    patient_classes=patient_classes,
    cohort_patients=patient_classes.index.tolist(),
    p_value_threshold=p_value_threshold,
)
original_by_label = select_motifs_by_label(
    discriminative_summary=original_summary,
    labels=labels,
    stringency=original_stringency,
    max_features=original_max_features,
)
original_all = set().union(*(set(v) for v in original_by_label.values()))

print(f'Original stringency: {original_stringency:.4f}')
print(f'Original max features per group: {original_max_features}')
{label: len(ids) for label, ids in original_by_label.items()}


In [ ]:
def _motif_list_text(motif_ids):
    return '|'.join(map(str, sorted(motif_ids, key=str)))

overlap_rows = []
for _, result_row in results.iterrows():
    sampled_patients = [p for p in str(result_row['sampled_patients']).split('|') if p and p != 'nan']
    other_label = result_row['other_label']
    other_patients = patient_classes.index[
        patient_classes[DiscriminativeMotifs.PATIENT_CLASS] == other_label
    ].tolist()
    cohort_patients = sampled_patients + other_patients
    stringency = float(result_row['optimal_stringency'])

    bootstrap_summary = discover_hard_discriminative_motifs(
        motifs_df=motifs_df,
        patient_classes=patient_classes,
        cohort_patients=cohort_patients,
        p_value_threshold=p_value_threshold,
    )
    bootstrap_by_label = select_motifs_by_label(
        discriminative_summary=bootstrap_summary,
        labels=labels,
        stringency=stringency,
        max_features=max_features,
    )

    label_rows = []
    for label in labels:
        bootstrap_set = set(bootstrap_by_label[label])
        original_set = set(original_by_label[label])
        retained = bootstrap_set & original_set
        added = bootstrap_set - original_set
        missing = original_set - bootstrap_set
        row = {
            'sampled_label': result_row['sampled_label'],
            'other_label': result_row['other_label'],
            'bootstrap_fraction': result_row['bootstrap_fraction'],
            'bootstrap_index': result_row['bootstrap_index'],
            'sampled_patients': result_row['sampled_patients'],
            'stringency': stringency,
            'motif_label': label,
            'original_selected_count': len(original_set),
            'bootstrap_selected_count': len(bootstrap_set),
            'retained_original_count': len(retained),
            'additional_count': len(added),
            'missing_original_count': len(missing),
            'retained_original_motifs': _motif_list_text(retained),
            'additional_motifs': _motif_list_text(added),
            'missing_original_motifs': _motif_list_text(missing),
        }
        label_rows.append(row)
        overlap_rows.append(row)

    bootstrap_all = set().union(*(set(v) for v in bootstrap_by_label.values()))
    retained_all = bootstrap_all & original_all
    added_all = bootstrap_all - original_all
    missing_all = original_all - bootstrap_all
    overlap_rows.append({
        'sampled_label': result_row['sampled_label'],
        'other_label': result_row['other_label'],
        'bootstrap_fraction': result_row['bootstrap_fraction'],
        'bootstrap_index': result_row['bootstrap_index'],
        'sampled_patients': result_row['sampled_patients'],
        'stringency': stringency,
        'motif_label': 'ALL',
        'original_selected_count': len(original_all),
        'bootstrap_selected_count': len(bootstrap_all),
        'retained_original_count': len(retained_all),
        'additional_count': len(added_all),
        'missing_original_count': len(missing_all),
        'retained_original_motifs': _motif_list_text(retained_all),
        'additional_motifs': _motif_list_text(added_all),
        'missing_original_motifs': _motif_list_text(missing_all),
    })

overlap = pd.DataFrame(overlap_rows)
overlap_path = OUTPUT_DIR / 'bootstrap_original_feature_overlap.csv'
overlap.to_csv(overlap_path, index=False)
print(f'Saved overlap table: {overlap_path}')
display(overlap)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_df = (
    results.groupby(["sampled_label", "bootstrap_fraction"], as_index=False)["optimal_mean_selected_motifs"]
    .mean()
    .sort_values("bootstrap_fraction")
)

if sns is not None:
    sns.lineplot(
        data=plot_df,
        x="bootstrap_fraction",
        y="optimal_mean_selected_motifs",
        hue="sampled_label",
        marker="o",
        ax=ax,
    )
else:
    for sampled_label, group in plot_df.groupby("sampled_label"):
        ax.plot(group["bootstrap_fraction"], group["optimal_mean_selected_motifs"], marker="o", label=sampled_label)

ax.set_xlabel("Bootstrap fraction")
ax.set_ylabel("Mean bootstrap selected count")
ax.set_title("Mean selected motifs vs bootstrap fraction")
ax.legend(title="Sampled label")
plt.show()

In [ ]:
overlap[['retained_original_motifs','missing_original_motifs','additional_motifs']].head(12)

In [ ]:
plot_df = overlap[overlap['motif_label'] != 'ALL'].copy()
fig, ax = plt.subplots(figsize=(9, 5))
if sns is not None:
    sns.scatterplot(
        data=plot_df,
        x='bootstrap_fraction',
        y='retained_original_count',
        hue='motif_label',
        style='sampled_label',
        size='additional_count',
        sizes=(30, 180),
        ax=ax,
    )
else:
    for label, group in plot_df.groupby('motif_label'):
        ax.scatter(group['bootstrap_fraction'], group['retained_original_count'], label=label)
ax.set_xlabel('Fraction of sampled-label patients kept')
ax.set_ylabel('Original motifs retained')
ax.set_title('Original feature retention; marker size reflects additional motifs')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()


## Fold-Union Original Motif Usage

This section matches the intended comparison: first build the no-bootstrap original LOOCV fold-union motif set using the original `best_feature_conf`, then ask which of those original fold-union motifs appear in each bootstrap's own fold-union motif set at its optimal stringency. This does not rerun RF, LOOCV classification, or Optuna; it only repeats the motif extraction/thresholding needed to recover feature identities.

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = REPO_ROOT
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.cohort_imbalance.imbalanced_bootstrap_stringency import (
    DEFAULT_ARTIFACT_PATH,
    discover_hard_discriminative_motifs,
    load_artifact,
    load_patient_classes,
    select_motifs_by_label,
)
from cism.cism import DiscriminativeMotifs

config_path = OUTPUT_DIR / 'imbalanced_bootstrap_config.json'
config = json.loads(config_path.read_text()) if config_path.exists() else {}
labels = config.get('labels', ['NN', 'NP'])
max_features = int(config.get('max_features', 30))
p_value_threshold = float(config.get('fanmod_p_value_threshold', 0.05))
artifact_path = Path(config.get('artifact_path', DEFAULT_ARTIFACT_PATH))

artifact = load_artifact(artifact_path)
discriminator = artifact['discriminator']
patient_classes = load_patient_classes(discriminator, labels)
motifs_df = discriminator.cism.motifs_dataset
best_feature_conf = artifact.get('best_feature_conf')
original_stringency = float(getattr(best_feature_conf, 'shared_fractionage', config.get('original_stringency', 0.4404145064107466)))
original_max_features = int(getattr(best_feature_conf, 'max_class_features', max_features))

def fold_union_motifs_by_label(cohort_patients, stringency, max_features_per_label):
    union_by_label = {label: set() for label in labels}
    for held_out_patient in cohort_patients:
        train_patients = [patient for patient in cohort_patients if patient != held_out_patient]
        train_classes = patient_classes.loc[train_patients, DiscriminativeMotifs.PATIENT_CLASS]
        if train_classes.nunique() < 2:
            continue
        fold_summary = discover_hard_discriminative_motifs(
            motifs_df=motifs_df,
            patient_classes=patient_classes,
            cohort_patients=train_patients,
            p_value_threshold=p_value_threshold,
        )
        selected_by_label = select_motifs_by_label(
            discriminative_summary=fold_summary,
            labels=labels,
            stringency=stringency,
            max_features=max_features_per_label,
        )
        for label, motif_ids in selected_by_label.items():
            union_by_label[label].update(motif_ids)
    return union_by_label

original_fold_union_by_label = fold_union_motifs_by_label(
    cohort_patients=patient_classes.index.tolist(),
    stringency=original_stringency,
    max_features_per_label=original_max_features,
)
original_fold_union_all = set().union(*(motifs for motifs in original_fold_union_by_label.values()))
original_motif_ids = sorted(original_fold_union_all, key=str)

print(f'Original stringency: {original_stringency:.4f}')
print(f'Original max features per group: {original_max_features}')
print({label: len(motifs) for label, motifs in original_fold_union_by_label.items()})
print(f'Original fold-union motifs total: {len(original_fold_union_all)}')


In [ ]:
usage_rows = []
for _, result_row in results.iterrows():
    sampled_patients = [p for p in str(result_row['sampled_patients']).split('|') if p and p != 'nan']
    other_label = result_row['other_label']
    other_patients = patient_classes.index[
        patient_classes[DiscriminativeMotifs.PATIENT_CLASS] == other_label
    ].tolist()
    cohort_patients = sampled_patients + other_patients
    bootstrap_stringency = float(result_row['optimal_stringency'])

    bootstrap_union_by_label = fold_union_motifs_by_label(
        cohort_patients=cohort_patients,
        stringency=bootstrap_stringency,
        max_features_per_label=max_features,
    )
    bootstrap_union_all = set().union(*(motifs for motifs in bootstrap_union_by_label.values()))

    original_nn = original_fold_union_by_label.get('NN', set())
    original_np = original_fold_union_by_label.get('NP', set())

    row = {
        'sampled_label': result_row['sampled_label'],
        'bootstrap_fraction': result_row['bootstrap_fraction'],
        'bootstrap_index': result_row['bootstrap_index'],
        'sampled_patients': result_row['sampled_patients'],
        'selected_stringency': bootstrap_stringency,
        'fold_union_motifs_cleared_count': len(bootstrap_union_all),
        'fold_union_motifs_discriminative_for_NN': len(bootstrap_union_by_label.get('NN', set())),
        'fold_union_motifs_discriminative_for_NP': len(bootstrap_union_by_label.get('NP', set())),
        'fraction_original_NN_motifs_used': (
            len(original_nn & bootstrap_union_all) / len(original_nn) if original_nn else float('nan')
        ),
        'fraction_original_NP_motifs_used': (
            len(original_np & bootstrap_union_all) / len(original_np) if original_np else float('nan')
        ),
        'fraction_all_original_motifs_used': (
            len(original_fold_union_all & bootstrap_union_all) / len(original_fold_union_all)
            if original_fold_union_all else float('nan')
        ),
    }
    for motif_id in original_motif_ids:
        row[f'original_motif_{motif_id}'] = motif_id in bootstrap_union_all
    usage_rows.append(row)

original_usage = pd.DataFrame(usage_rows)
usage_path = OUTPUT_DIR / 'bootstrap_original_fold_union_motif_usage.csv'
original_usage.to_csv(usage_path, index=False)
print(f'Saved original fold-union motif usage table: {usage_path}')
display(original_usage.head())


In [ ]:
summary_cols = [
    'sampled_label',
    'bootstrap_fraction',
    'bootstrap_index',
    'selected_stringency',
    'fold_union_motifs_cleared_count',
    'fold_union_motifs_discriminative_for_NN',
    'fold_union_motifs_discriminative_for_NP',
    'fraction_original_NN_motifs_used',
    'fraction_original_NP_motifs_used',
    'fraction_all_original_motifs_used',
]
display(original_usage[summary_cols].sort_values(['sampled_label', 'bootstrap_fraction', 'bootstrap_index']))

fig, ax = plt.subplots(figsize=(9, 5))
for sampled_label, group in original_usage.groupby('sampled_label'):
    group = group.sort_values('bootstrap_fraction')
    ax.scatter(
        group['bootstrap_fraction'],
        group['fraction_all_original_motifs_used'],
        label=f'sampled {sampled_label}',
        alpha=0.75,
    )
ax.set_xlabel('Fraction of sampled-label patients kept')
ax.set_ylabel('Fraction of original fold-union motifs used')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Original fold-union motif reuse by bootstrap')
ax.legend(title='Experiment')
plt.show()
